In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [14]:
import pandas as pd

from src.pose import extract_pose
from src.preprocessing import normalize_pose, apply_smoothing
from src.features import extract_features
from src.aggregate import aggregate_features
import joblib


def predict_video(video_path):
    # Load model + feature order
    data = joblib.load("pushup_model.pkl")
    model = data["model"]
    feature_names = data["features"]

    # Run pipeline
    pose_df = extract_pose(video_path)
    norm_df = normalize_pose(pose_df)
    smooth_df = apply_smoothing(norm_df)
    features_df = extract_features(smooth_df)
    aggregated = aggregate_features(features_df)

    # Convert to DataFrame
    X = pd.DataFrame([aggregated])

    # 🔥 CRITICAL: align feature order
    X = X[feature_names]

    # Predict
    pred = model.predict(X)[0]
    prob = model.predict_proba(X)[0]

    print(f"\nPrediction: {'Correct' if pred==1 else 'Incorrect'}")
    print(f"Confidence: {max(prob):.2f}")

    return pred

In [13]:
predict_video("../data/push-up/cor5.mp4")

Processing video...


100%|██████████| 115/115 [00:02<00:00, 52.94it/s]



Prediction: Incorrect
Confidence: 0.57


np.int64(0)

In [ ]:
predict_video("../data/push-up/push-up_24")

Processing video...


100%|██████████| 125/125 [00:02<00:00, 53.94it/s]



Prediction: Incorrect
Confidence: 1.00


np.int64(0)

In [15]:
import numpy as np
def simulate_action(state, action):
    new_state = state.copy()

    success_prob = 0.7  # user follows instruction

    if np.random.rand() < success_prob:

        if action == 0:  # go deeper
            improvement = np.random.uniform(0.02, 0.08)
            new_state["max_depth"] += improvement

        elif action == 1:  # fix elbow
            improvement = np.random.uniform(2, 6)
            new_state["min_elbow_angle"] -= improvement

        elif action == 2:
            new_state["avg_alignment"] += np.random.uniform(1, 3)

        elif action == 3:
            new_state["min_hip_angle"] += np.random.uniform(1, 4)

    else:
        # user fails / worsens slightly
        new_state += np.random.normal(0, 0.02, size=len(state))

    return new_state

In [ ]:
def get_score(self, state):
    X = pd.DataFrame([state], columns=self.feature_names)
    X = X[self.feature_names]

    prob = self.model.predict_proba(X)[0][1]

    # Safety: keep within valid bounds
    prob = np.clip(prob, 0.0, 1.0)

    return float(prob)

In [17]:
class PushupEnv:
    def __init__(self, model, feature_names):
        self.model = model
        self.feature_names = feature_names
        self.max_steps = 10

    def reset(self):
        self.state = self.sample_initial_state()
        self.steps = 0
        return self.state

    def step(self, action):
        prev_score = self.get_score(self.state)

        next_state = simulate_action(self.state, action)

        curr_score = self.get_score(next_state)

        reward = curr_score - prev_score

        self.state = next_state
        self.steps += 1

        done = self.steps >= self.max_steps or curr_score > 0.9

        return next_state, reward, done, {}

    def get_score(self, state):
        X = pd.DataFrame([state], columns=self.feature_names)
        return self.model.predict_proba(X)[0][1]

In [ ]:
data = joblib.load("pushup_model.pkl")
model = data["model"]
feature_names = data["features"]